## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-18-04-46-06 +0000,nypost,Mary Peltola finally declared winner of Alaska...,https://nypost.com/2026/11/17/us-news/mary-pel...
1,2026-11-18-04-37-42 +0000,nypost,Yosemite issues urgent travel warning as sever...,https://nypost.com/2026/11/17/us-news/yosemite...
2,2026-11-18-04-36-00 +0000,wsj,"Iran Attacks Kuwait and Bahrain, Further Escal...",https://www.wsj.com/world/middle-east/iran-att...
3,2026-11-18-04-34-39 +0000,bbc,Democrats win Alaska after Trump says voters c...,https://www.bbc.co.uk/news/articles/cxnv4jlr26...
4,2026-11-18-04-33-50 +0000,nyt,Live Updates: Trump Meets Philippine Leader Be...,https://www.nytimes.com/live/2026/11/17/world/...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2546/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
32,trump,60
131,nvidia,34
136,huang,25
117,new,24
325,up,22


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
265,2026-11-17-18-34-00 +0000,wsj,A lawyer for Fed governor Lisa Cook said she w...,https://www.wsj.com/politics/policy/feds-lisa-...,158
385,2026-11-17-12-57-00 +0000,wsj,Nvidia formed a board committee to oversee nat...,https://www.wsj.com/politics/national-security...,155
114,2026-11-17-23-02-12 +0000,bbc,"As Trump and Xi meet, China's AI dreams face a...",https://www.bbc.co.uk/news/articles/c5vg23g92r...,155
271,2026-11-17-18-24-21 +0000,bbc,Jensen Huang says he would be 'honoured' to re...,https://www.bbc.co.uk/news/articles/cwx2en2vrx...,135
137,2026-11-17-22-33-00 +0000,wsj,"Hung Cao, President Trump’s pick to lead the N...",https://www.wsj.com/politics/national-security...,130


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
265,158,2026-11-17-18-34-00 +0000,wsj,A lawyer for Fed governor Lisa Cook said she w...,https://www.wsj.com/politics/policy/feds-lisa-...
385,155,2026-11-17-12-57-00 +0000,wsj,Nvidia formed a board committee to oversee nat...,https://www.wsj.com/politics/national-security...
227,90,2026-11-17-20-01-00 +0000,wsj,Progressive activists say the Senate next year...,https://www.wsj.com/politics/policy/the-angry-...
360,65,2026-11-17-14-25-45 +0000,bbc,How new US chip restrictions on South East Asi...,https://www.bbc.co.uk/news/articles/cwq6e2qgvm...
137,62,2026-11-17-22-33-00 +0000,wsj,"Hung Cao, President Trump’s pick to lead the N...",https://www.wsj.com/politics/national-security...
218,61,2026-11-17-20-31-11 +0000,nyt,Ukraine Talks Run Up Against Deep Enmity Betwe...,https://www.nytimes.com/2026/11/17/us/politics...
383,55,2026-11-17-13-07-05 +0000,cbc,Iran and U.S. trade fire as Iran denies report...,https://www.cbc.ca/news/world/iran-us-war-bahr...
7,51,2026-11-18-04-23-00 +0000,wsj,Democrat Mary Peltola unseated Republican Sen....,https://www.wsj.com/politics/elections/democra...
423,48,2026-11-17-11-00-00 +0000,latimes,Hiltzik: If Jensen Huang really was 'irreplace...,https://www.latimes.com/business/story/2026-11...
111,48,2026-11-17-23-13-17 +0000,nypost,Why tech firms you’ve never heard of are causi...,https://nypost.com/2026/11/17/business/why-tec...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
